In [1]:
import joblib
import os
import h5py
import torch
import numpy as np
import pandas as pd

In [ ]:
BASE_DIR = "/Users/jk/Documents/Lab2/visium/python_analysis/cpath/trident/wsi_svs/trident_processed_200px/20x_200px_0px_overlap/features_uni_v2"
OUTPUT_PATH = "/Users/jk/Documents/Lab2/visium/python_analysis/cpath/trident/wsi_svs/all_sample_proportions_200px.csv"

In [3]:
clf = joblib.load('/Users/jk/Documents/Lab2/visium/python_analysis/cpath/logistic_classifier/niche_classifier_final_uni2_100um.joblib')


In [4]:
all_prop_rows = []

h5s = [h5 for h5 in os.listdir(BASE_DIR) if h5.endswith(".h5") and not h5.startswith(".")]

for h5 in h5s:
    library_id = h5.removesuffix(".h5")
    
    print(f"\n🔹 Processing {library_id}")
          
    with h5py.File(os.path.join(BASE_DIR, h5), "r") as f:
        features = torch.from_numpy(f["features"][:])

    preds = clf.predict(features)
    labels, counts = np.unique(preds, return_counts=True)
    proportions = counts / counts.sum()

    # Build row dictionary
    row = {"library_id": library_id} 
    for label, prop in zip(labels, proportions):
        row[f"class_{label}"] = prop
    all_prop_rows.append(row)

# -------------------------------
# Combine and export
# -------------------------------
prop_df_all = pd.DataFrame(all_prop_rows).fillna(0)

# Sort columns: metadata first, then classes
cols = ["library_id"] + sorted([c for c in prop_df_all.columns if c.startswith("class_")])
prop_df_all = prop_df_all[cols]

prop_df_all.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ Saved combined table → {OUTPUT_PATH}")
print(prop_df_all.head())


🔹 Processing 08-38774-B2

🔹 Processing 21-24095-A3

🔹 Processing 22-16220-B1

🔹 Processing 18-57617-A1

🔹 Processing 20-28197-A1

🔹 Processing 19-18542-A4

🔹 Processing 19-35057-C3

🔹 Processing 20-33940-B2

🔹 Processing 20-41615-B1

🔹 Processing 24-27523-C5

🔹 Processing 20-22642-A1

🔹 Processing 20-24241-A2

🔹 Processing 20-26330-B3

🔹 Processing 20-17688-B2

🔹 Processing 06-30914-A1

🔹 Processing 23-15209-A3

🔹 Processing 20-03948-B3

🔹 Processing 21-55747-C3

🔹 Processing 21-06301-B2

🔹 Processing 21-24837-A1

🔹 Processing 23-50343-B2

🔹 Processing 20-41501-C1

🔹 Processing 17-25789-B1

🔹 Processing 23-41922-B2

🔹 Processing 21-57231-A3

🔹 Processing 16-39724-B1

🔹 Processing 22-18440-A2

🔹 Processing 20-33362-C4

✅ Saved combined table → /Users/jk/Documents/Lab2/visium/python_analysis/cpath/trident/wsi_svs/all_sample_proportions_200px.csv
    library_id   class_0   class_1   class_2   class_3   class_4   class_5
0  08-38774-B2  0.186323  0.194245  0.294479  0.016409  0.059736  0.

In [5]:
prop_df_all.shape

(28, 7)